In [1]:
import pandas as pd
df = pd.read_csv("PC_Uncertainty_dataset.csv")

In [2]:
df

,Gross Electrical Output (MWg),As-Delivered Primary Fuel Cost ($/ton),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg),Important Performance & Cost Factors: CO2 Removed (tons/hr),Important Performance & Cost Factors: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)
0,1534.0,48.900002,38.299999,60.877499,3.6775,868.279724,1336.483643,159.369995
1,2122.0,45.500000,35.700001,87.982498,4.8325,1722.348755,1825.054810,110.072769
2,1498.0,67.300003,42.900002,62.437500,1.5075,869.229858,1337.637817,166.645416
3,418.0,31.700001,28.700001,89.542503,4.5175,345.206390,360.549957,139.354156
4,1666.0,57.500000,11.500000,64.387497,1.5775,996.329956,1486.579224,159.967117
...,...,...,...,...,...,...,...,...
195,202.0,58.099998,38.700001,70.432503,2.4175,131.969788,178.349472,223.930115
196,1462.0,61.500000,31.100000,87.592499,5.1475,1181.545776,1252.625000,122.992653
197,646.0,60.900002,48.700001,66.142502,4.4475,396.695221,557.657959,178.050934
198,262.0,37.900002,29.900000,62.827499,7.4925,153.009171,218.053696,219.787796


In [3]:
df.isna().sum()

Gross Electrical Output (MWg)                                       0
As-Delivered Primary Fuel Cost ($/ton)                              0
Cooling Water Temperature Rise (°F)                                 0
Absorber CO2 Removal Efficiency (%)                                 0
Amine Scrubber Power Requirement (% MWg)                            0
Important Performance & Cost Factors: CO2 Removed (tons/hr)         0
Important Performance & Cost Factors: Net Electrical Output (MW)    0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)       0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Gross Electrical Output (MWg)',
    'As-Delivered Primary Fuel Cost ($/ton)',
    'Cooling Water Temperature Rise (°F)',
    'Absorber CO2 Removal Efficiency (%)',
    'Amine Scrubber Power Requirement (% MWg)']]

y = df['Important Performance & Cost Factors: Net Electrical Output (MW)']

In [6]:
x

,Gross Electrical Output (MWg),As-Delivered Primary Fuel Cost ($/ton),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg)
0,1534.0,48.900002,38.299999,60.877499,3.6775
1,2122.0,45.500000,35.700001,87.982498,4.8325
2,1498.0,67.300003,42.900002,62.437500,1.5075
3,418.0,31.700001,28.700001,89.542503,4.5175
4,1666.0,57.500000,11.500000,64.387497,1.5775
...,...,...,...,...,...
195,202.0,58.099998,38.700001,70.432503,2.4175
196,1462.0,61.500000,31.100000,87.592499,5.1475
197,646.0,60.900002,48.700001,66.142502,4.4475
198,262.0,37.900002,29.900000,62.827499,7.4925


In [7]:
y

0      1336.483643
1      1825.054810
2      1337.637817
3       360.549957
4      1486.579224
          ...     
195     178.349472
196    1252.625000
197     557.657959
198     218.053696
199     542.019653
Name: Important Performance & Cost Factors: Net Electrical Output (MW), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== FINAL PYOMO MODEL ===\n")


=== FINAL PYOMO MODEL ===



In [9]:
from sklearn.preprocessing import PolynomialFeatures

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 3}
Train R2: 0.9999999999497956
Test R2: 0.99999999955918


In [10]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Polynomial | RMSE: 0.0134 | MAPE: 0.0032%


In [11]:
import numpy as np, json, joblib

joblib.dump(best_poly, "PC_Net_Output_poly.pkl")

poly   = best_poly.named_steps['poly']
scaler = best_poly.named_steps['scaler']
reg    = best_poly.named_steps['reg']

coef_real      = reg.coef_ / scaler.scale_
intercept_real = reg.intercept_ - np.sum(reg.coef_ * scaler.mean_ / scaler.scale_)

feature_names = list(x.columns)
powers = poly.powers_

terms = [{"coef": float(c),
          "powers": {feature_names[j]: int(p[j]) for j in range(len(feature_names)) if p[j] > 0}}
         for p, c in zip(powers, coef_real)]

equation = {"intercept": float(intercept_real), "degree": int(poly.degree), "terms": terms}

P = poly.transform(x_test.values)
assert np.allclose(intercept_real + P @ coef_real, best_poly.predict(x_test)), "mismatch!"
print("poly equation matches pipeline ✓  terms:", len(terms))

json.dump(equation, open("PC_Net_Output_poly_eq.json", "w"), indent=2)

poly equation matches pipeline ✓  terms: 55


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(
